In [1]:
# All Necessary Imports
import numpy as np
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import re
from textblob import TextBlob
from wordcloud import WordCloud
import seaborn as sns
import matplotlib.pyplot as plt
import cufflinks as cf
%matplotlib inline
from plotly.offline import init_notebook_mode, iplot
init_notebook_mode(connected = True)
cf.go_offline();
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns",None)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense, Dropout # Added Dropout

# Download NLTK data (run this only once)
nltk.download('stopwords')
nltk.download('wordnet')

# Load and Prepare the Data
df = pd.read_csv("amazon.csv")
df = df.sort_values("wilson_lower_bound", ascending=False)
df.drop("Unnamed: 0", inplace=True, axis=1)

# Advanced Data Cleaning and Column Creation
def advanced_clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower()
    words = text.split()
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return " ".join(cleaned_words)

df['reviewText_cleaned'] = df['reviewText'].apply(advanced_clean_text)
df['sentiment_class'] = df['overall'].apply(lambda x: 'Positive' if x >= 4 else 'Negative' if x <= 2 else 'Neutral')
print("Initial Data Cleaning and Preparation Complete!")

# Deep Learning Preprocessing
# Encode the target variable from string labels to integers
le = LabelEncoder()
df['sentiment_encoded'] = le.fit_transform(df['sentiment_class'])

# Select features (X) and target (y)
X_nn = df['reviewText_cleaned']
y_nn = df['sentiment_encoded']

# Split data into training and testing sets
X_train_nn, X_test_nn, y_train_nn, y_test_nn = train_test_split(X_nn, y_nn, test_size=0.2, random_state=42, stratify=y_nn)

# Tokenize the text
tokenizer = Tokenizer(num_words=5000, oov_token="<unk>")
tokenizer.fit_on_texts(X_train_nn)
X_train_seq = tokenizer.texts_to_sequences(X_train_nn)
X_test_seq = tokenizer.texts_to_sequences(X_test_nn)

# Pad sequences to a fixed length
max_len = 100
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

print("Data preprocessed for deep learning!")
print(f"Shape of training data: {X_train_pad.shape}")
print(f"Shape of testing data: {X_test_pad.shape}")

# Building and Training the Neural Network
num_classes = len(le.classes_)
model = Sequential([
    Embedding(input_dim=5000, output_dim=16, input_length=max_len),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5), # Add a Dropout layer with a rate of 0.5 (50%)
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print("\nModel Summary:")
model.summary()

history = model.fit(X_train_pad, y_train_nn, epochs=10, validation_data=(X_test_pad, y_test_nn), verbose=1)

print("\nTensorFlow model with Dropout trained successfully!")

# Model Evaluation
y_pred_probs = model.predict(X_test_pad)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

# Decode predictions back to original labels
y_test_labels = le.inverse_transform(y_test_nn)
y_pred_labels = le.inverse_transform(y_pred_classes)

# Print the classification report
print("\nTensorFlow Model Classification Report:")
print(classification_report(y_test_labels, y_pred_labels))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sathv\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sathv\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Initial Data Cleaning and Preparation Complete!
Data preprocessed for deep learning!
Shape of training data: (3932, 100)
Shape of testing data: (983, 100)

Model Summary:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8993 - loss: 0.3853 - val_accuracy: 0.9054 - val_loss: 0.3159
Epoch 2/10
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9128 - loss: 0.2787 - val_accuracy: 0.9176 - val_loss: 0.2557
Epoch 3/10
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9451 - loss: 0.1777 - val_accuracy: 0.9278 - val_loss: 0.2288
Epoch 4/10
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9682 - loss: 0.1044 - val_accuracy: 0.9207 - val_loss: 0.2597
Epoch 5/10
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9822 - loss: 0.0596 - val_accuracy: 0.9207 - val_loss: 0.2986
Epoch 6/10
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9929 - loss: 0.0280 - val_accuracy: 0.9237 - val_loss: 0.3217
Epoch 7/10
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9980 - loss: 0.0137 - val_accuracy: 0.9207 - val_loss: 0.3520
Epoch 8/10
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9995 - loss: 0.0070 - val_accuracy: 0.